# F3 — Núcleo algorítmico, eficiencia y orientación a objetos

**Proyecto:** precios cotizados de Airbnb en Buenos Aires.

La fase reorganiza el preprocesamiento de F2 mediante clases y compara dos formas
de calcular los mismos resúmenes descriptivos. Se mantienen la pregunta, la fuente,
la moneda ARS, las reglas de elegibilidad y las condiciones de estancia de F2.
No se incorporan objetivos predictivos ni se modifica el archivo original.

El código reutilizable reside en `src/f3_transformadores.py`, `src/f3_pipeline.py`,
`src/f3_medicion.py` y `src/f3_validacion.py`; el cuaderno los importa.
El diseño sigue la guía sumativa y el apunte de Fase 3 (Universidad Andrés Bello, s. f.-a, s. f.-b).

## 1. Entorno y condiciones de ejecución

Se requiere la `.venv` del proyecto, las cinco dependencias fijadas, el original
local y los tres resultados de F2 con sus huellas registradas. La comprobación se
detiene si faltan archivos o si alguno no coincide con la evidencia de F2.
Los datos por anuncio permanecen excluidos de Git; únicamente se publican agregados.

In [1]:
from pathlib import Path
import json
import platform
import re
import subprocess
import sys

import numpy as np
import pandas as pd
from IPython.display import display

candidatas = [Path.cwd().resolve(), Path.cwd().resolve().parent]
RAIZ = next((p for p in candidatas if (p / 'src/preprocesamiento.py').is_file()
             and (p / 'requirements.txt').is_file()), None)
if RAIZ is None:
    raise RuntimeError('Abrir el proyecto desde su raíz o desde F3/.')
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src import preprocesamiento as f2
from src.reconocimiento import (calcular_sha256, cargar_dataset, consultar_git,
                                huella_tabla, verificar_entorno)
from src.f3_pipeline import construir_pipeline
from src.f3_transformadores import SelectorGrupoComparable, VerificadorCotizaciones
from src.f3_medicion import (comparar_implementaciones, comprobar_resumenes,
                             resumen_por_bucle, resumen_vectorizado)
from src.f3_validacion import cargar_referencias_f2, comprobar_equivalencia

DESTINO = RAIZ / 'docs/evidencias/f3'
RUTA_EVIDENCIA_F2 = RAIZ / 'docs/evidencias/preprocesamiento_f2.json'
AMBITO_EJECUCION = 'Notebook ejecutado en el entorno local del proyecto.'
DESTINO.mkdir(parents=True, exist_ok=True)
entorno, dependencias = verificar_entorno(RAIZ)
valores = entorno.set_index('componente')['valor']
assert bool(valores['Kernel en .venv del proyecto']), 'Seleccionar el kernel .venv del proyecto.'
assert bool(valores['Python de la serie 3.13']), 'Se requiere Python de la serie 3.13.'
assert bool(valores['pip check correcto']) and dependencias['cumple_declaracion'].all()
git_inicio = consultar_git(RAIZ)
assert git_inicio['disponible'], 'No se pudo consultar Git desde la raíz.'
for ruta in ['data/raw/listings.csv.gz', 'data/processed/comprobacion_f3.csv']:
    verificacion = subprocess.run(['git', '-C', str(RAIZ), 'check-ignore', '-q', '--', ruta],
                                  capture_output=True)
    assert verificacion.returncode == 0, f'La ruta local {ruta} debe estar excluida de Git.'
rastreados = subprocess.run(['git', '-C', str(RAIZ), 'ls-files', '--', 'data/raw/', 'data/processed/'],
                            capture_output=True, text=True, encoding='utf-8', check=True)
assert not [p for p in rastreados.stdout.splitlines() if Path(p).name != '.gitkeep'], 'Hay datos rastreados por Git.'
display(entorno, dependencias, pd.DataFrame([git_inicio]))

,componente,valor
0,Python,3.13.17
1,Intérprete del kernel,c:\Users\devco\Documents\Magister\proyecto-mcd...
2,Plataforma,Windows-11-10.0.26300-SP0
3,Entorno virtual activo,True
4,Kernel en .venv del proyecto,True
5,Python de la serie 3.13,True
6,pip check correcto,True
7,Salida de pip check,No broken requirements found.


,dependencia,declaracion,version_instalada,importacion_correcta,version_fijada,cumple_declaracion,observacion
0,ipykernel,ipykernel==7.4.0,7.4.0,True,True,True,Versión fijada
1,jupyterlab,jupyterlab==4.6.6,4.6.6,True,True,True,Versión fijada
2,notebook,notebook==7.6.5,7.6.5,True,True,True,Versión fijada
3,numpy,numpy==2.5.4,2.5.4,True,True,True,Versión fijada
4,pandas,pandas==3.1.0,3.1.0,True,True,True,Versión fijada


,disponible,rama,commit,cambios_sin_confirmar
0,True,feature/f3-nucleo-algoritmico,06a4c5a66b5cd7b59e463c4144b0cc66dcae3643,True


## 2. Lectura de la fuente y de los resultados de F2

El original se utiliza para verificar la reorganización del preprocesamiento; los
CSV de F2 constituyen la referencia previamente guardada. Sus huellas se cotejan
antes de comparar valores. No se descargan archivos ni se vuelve a ejecutar F2.

In [2]:
RUTA_ORIGINAL = RAIZ / 'data/raw/listings.csv.gz'
sha_original = calcular_sha256(RUTA_ORIGINAL)
assert sha_original == f2.SHA256_ESPERADO, 'La fuente no coincide con la huella documentada.'
datos = cargar_dataset(RUTA_ORIGINAL)
huella_original = huella_tabla(datos)
referencias, huellas_f2 = cargar_referencias_f2(RAIZ, RUTA_EVIDENCIA_F2)
display(pd.DataFrame([{'filas_originales': len(datos), 'columnas_originales': datos.shape[1],
                       'sha256_verificado': sha_original}]), huellas_f2)

,filas_originales,columnas_originales,sha256_verificado
0,29753,90,c8db6461c06c0de3efbdda6821d6d28c9aef7dff3c6fde...


,resultado_f2,sha256_coincide,sha256,filas
0,base,True,fea1b7af431c363145ea04d34bb7201a3f53a9a7cbde47...,27633
1,principal,True,a4a80c6246dd884efde0916b2d57b6047a8a283e3bcf61...,2080
2,sensibilidad,True,850a74335b16942a5a6fcb923fbf3c84302172a390e732...,5039


## 3. Clases, estado y composición del flujo

`TransformadorCotizaciones` define el contrato común `ajustar()`/`transformar()`:
valida entradas, protege el estado y entrega copias de sus parámetros.
Las tres clases hijas especializan la verificación, la base analítica y la selección
de condiciones comparables. `PipelineCotizaciones` las recorre por la misma interfaz,
sin preguntar su tipo: se aplican herencia, encapsulamiento y polimorfismo
(Python Software Foundation, s. f.-a).

El patrón **método plantilla (Template Method)** concentra el control de estado en
la clase base (Gamma et al., 1994). Sin él, cada etapa repetiría las mismas validaciones. La composición
del flujo permite sustituir etapas sin modificar el bucle del orquestador.

El estudio es descriptivo: no requiere una partición artificial de entrenamiento y
prueba. Los límites IQR y la ventana se ajustan sobre la base descriptiva declarada
y permanecen fijos al transformar; si se incorpora predicción, el ajuste deberá
limitarse al entrenamiento. Los faltantes accesorios se conservan, como en F2.

In [3]:
pipeline = construir_pipeline().ajustar(datos)
etapas = pipeline.transformar_etapas(datos)
base = etapas['PreparadorBaseAnalitica']
principal = etapas['SelectorGrupoComparable']
selector = SelectorGrupoComparable().ajustar(base)
sensibilidad = selector.ampliar_al_mes(base)
parametros_antes = pipeline.parametros
display(pd.DataFrame(pipeline.registro_ajuste))
display(pd.DataFrame([{'base_elegible': len(base), 'grupo_principal': len(principal),
                       'grupo_adicional': len(sensibilidad),
                       'entrada_principal': selector.parametros['entrada'],
                       'duracion_noches': selector.parametros['noches']}]))
display(pd.DataFrame([{'etapa': nombre, 'parametros': json.dumps(p, ensure_ascii=False)}
                      for nombre, p in pipeline.parametros.items()]))

,etapa,filas_entrada,filas_salida,columnas_salida
0,VerificadorCotizaciones,29753,29753,31
1,PreparadorBaseAnalitica,29753,27633,21
2,SelectorGrupoComparable,27633,2080,21


,base_elegible,grupo_principal,grupo_adicional,entrada_principal,duracion_noches
0,27633,2080,5039,2026-09-27,1


,etapa,parametros
0,VerificadorCotizaciones,"{""columnas_requeridas"": [""id"", ""host_id"", ""las..."
1,PreparadorBaseAnalitica,"{""limites_iqr"": {""Entire home/apt"": {""inferior..."
2,SelectorGrupoComparable,"{""entrada"": ""2026-09-27"", ""mes"": ""2026-09"", ""n..."


## 4. Equivalencia entre F2 y F3

Se exige **igualdad estricta en memoria** entre la vista/base de F2 y F3, incluyendo
tipos, orden y valores. Los CSV releídos se cotejan por separado con las tolerancias
de F2 (`rtol=1e-12`, `atol=1e-7`) y sin exigir tipos idénticos tras serializar a texto.
Las dos comprobaciones tienen distinto alcance y se registran explícitamente
(The pandas development team, s. f.). La fuente y la tabla de entrada deben permanecer
intactas. Las fechas comparables y la sensibilidad mensual mantienen el alcance de F2;
no se reinterpretan como ajuste por inflación ni como muestra representativa.

In [4]:
equivalencia = comprobar_equivalencia(datos, etapas, referencias, sensibilidad)
validacion = f2.validar_base(base)
assert calcular_sha256(RUTA_ORIGINAL) == sha_original
assert huella_tabla(datos) == huella_original
assert pipeline.parametros == parametros_antes
validacion = pd.concat([validacion, equivalencia, pd.DataFrame([
    {'control': 'archivo_original_inmutable', 'cumple': True},
    {'control': 'tabla_original_inmutable', 'cumple': True},
    {'control': 'parametros_estables_al_transformar', 'cumple': True},
])], ignore_index=True)
display(validacion)
faltantes = pd.DataFrame([
    {'variable': c, 'n_faltantes': int(base[c].isna().sum()),
     'porcentaje_faltantes': 100 * float(base[c].isna().mean()), 'tipo': str(base[c].dtype)}
    for c in base.columns
])
_, cambios_complementarios = f2.preparar_base(etapas['VerificadorCotizaciones'])
transformaciones = pd.DataFrame([
    {'transformacion': 'Casting de capacidad y noches a enteros', 'filas': len(base),
     'justificacion': 'Conservar conteos y duraciones enteras verificadas.'},
    {'transformacion': 'Conservación de faltantes con indicadores', 'filas': int(base[['dormitorios', 'camas', 'valoracion']].isna().any(axis=1).sum()),
     'justificacion': 'Evitar inventar conteos y cambiar la cobertura de anuncios.'},
    {'transformacion': 'Logaritmo log1p del precio', 'filas': len(base),
     'justificacion': 'Escala auxiliar reversible; el precio principal conserva ARS.'},
    {'transformacion': 'Razón precio/capacidad', 'filas': len(base),
     'justificacion': 'Referencia por capacidad anunciada, sin interpretarla como cobro por persona.'},
    {'transformacion': 'Marcación IQR por tipo', 'filas': int(base['precio_atipico_iqr_tipo'].sum()),
     'justificacion': 'Señalar extremos sin retirarlos automáticamente.'},
])
display(faltantes, cambios_complementarios, transformaciones)

,control,cumple
0,base_no_vacia,True
1,id_unico_y_presente,True
2,precio_finito_positivo,True
3,moneda_ARS,True
4,fechas_y_duracion_coherentes,True
5,capacidad_entera_positiva,True
6,grupos_principales_presentes,True
7,tipos_reconocidos,True
8,log_precio_reversible,True
9,razon_por_plaza_coherente,True


,variable,n_faltantes,porcentaje_faltantes,tipo
0,id,0,0.000000,string
1,host_id,0,0.000000,string
2,fecha_extraccion,0,0.000000,datetime64[us]
3,barrio,0,0.000000,string
4,tipo_alojamiento,0,0.000000,string
5,capacidad,0,0.000000,int64
6,precio_ars_noche,0,0.000000,float64
7,total_cotizado_ars,0,0.000000,float64
8,moneda,0,0.000000,string
9,entrada,0,0.000000,datetime64[us]


,variable,faltantes_antes,invalidos_convertidos_a_faltante
0,dormitorios,13,0
1,camas,478,0
2,valoracion,3077,0


,transformacion,filas,justificacion
0,Casting de capacidad y noches a enteros,27633,Conservar conteos y duraciones enteras verific...
1,Conservación de faltantes con indicadores,3510,Evitar inventar conteos y cambiar la cobertura...
2,Logaritmo log1p del precio,27633,Escala auxiliar reversible; el precio principa...
3,Razón precio/capacidad,27633,"Referencia por capacidad anunciada, sin interp..."
4,Marcación IQR por tipo,2209,Señalar extremos sin retirarlos automáticamente.


## 5. Casos límite, excepciones y protección del estado

Se prueban tres escenarios: el conjunto real, una fila válida y entradas que deben
fallar con mensajes claros. El conjunto con una fila sirve únicamente como prueba
técnica; no acredita una referencia estable de precios. Las pruebas automatizadas
añaden faltantes totales, cambios de distribución y alteraciones deliberadas del resultado.

In [5]:
id_ejemplo = principal['id'].iloc[0]
una_fila = datos.loc[datos['id'].astype('string').eq(id_ejemplo)].copy()
assert len(una_fila) == 1
limite = construir_pipeline().ajustar(una_fila).transformar(una_fila)
assert len(limite) == 1
casos = [{'caso': 'normal: conjunto real', 'resultado': 'verificado'},
         {'caso': 'límite: una fila válida', 'resultado': 'verificado'}]
for descripcion, accion, error_esperado in [
    ('sin ajuste previo', lambda: VerificadorCotizaciones().transformar(datos), RuntimeError),
    ('columna requerida ausente', lambda: construir_pipeline().ajustar(datos.drop(columns='price')), ValueError),
    ('tipo de entrada incorrecto', lambda: construir_pipeline().ajustar(None), TypeError),
]:
    try:
        accion()
    except error_esperado:
        casos.append({'caso': descripcion, 'resultado': f'{error_esperado.__name__} capturada'})
    else:
        raise AssertionError(f'No se detectó el error esperado: {descripcion}')
externos = pipeline.parametros
externos['SelectorGrupoComparable']['noches'] = 999
assert pipeline.parametros == parametros_antes
casos = pd.DataFrame(casos)
display(casos)

,caso,resultado
0,normal: conjunto real,verificado
1,límite: una fila válida,verificado
2,sin ajuste previo,RuntimeError capturada
3,columna requerida ausente,ValueError capturada
4,tipo de entrada incorrecto,TypeError capturada


## 6. Dos implementaciones del mismo cálculo

Se comparan tamaño de grupo, mediana y cuartiles por barrio. La versión inicial
filtra la tabla una vez por barrio; la candidata agrupa una vez y calcula los
cuartiles mediante operaciones de pandas. Antes de medir, ambas deben devolver
los mismos grupos, conteos y orden. Los importes se cotejan con tolerancia absoluta
de `1e-7 ARS` y `rtol=0` por el redondeo binario de los cuartiles, sin redondear
precios antes de calcular. También se comparan con los resúmenes de F2.
No se optimiza una función a costa de cambiar la respuesta.

In [6]:
resumenes = {}
for variable in ['barrio', 'tipo_alojamiento', 'capacidad']:
    inicial = resumen_por_bucle(principal, variable)
    candidata = resumen_vectorizado(principal, variable)
    comprobar_resumenes(inicial, candidata)
    comprobar_resumenes(candidata, f2.resumir_grupos(principal, variable))
    resumenes[variable] = candidata
display(resumenes['tipo_alojamiento'], resumenes['barrio'].head(10))

,tipo_alojamiento,n,mediana_ARS,q1_ARS,q3_ARS,grupo_menor_30
0,Entire home/apt,1692,120800.0,98302.9475,166640.85,False
1,Hotel room,40,253784.0,199257.0000,298822.25,False
2,Private room,329,69000.0,47000.0000,107018.00,False
3,Shared room,19,35775.0,24431.0000,42654.00,True


,barrio,n,mediana_ARS,q1_ARS,q3_ARS,grupo_menor_30
0,Almagro,46,104724.50,61076.295,125363.25,False
1,Balvanera,78,87143.00,67382.750,117719.00,False
2,Barracas,15,95681.00,82333.500,118552.50,True
3,Belgrano,171,140650.95,101666.290,177954.03,False
4,Boca,5,128421.00,99373.000,163584.00,True
5,Boedo,2,50422.20,35913.100,64931.30,True
6,Caballito,17,102125.00,44336.000,116190.00,True
7,Chacarita,57,96316.00,79499.000,136791.00,False
8,Coghlan,7,92951.92,91767.205,141345.00,True
9,Colegiales,43,137440.44,93258.500,187280.00,False


## 7. Mediciones de tiempo y memoria

Se utiliza `timeit.repeat`: cinco repeticiones y tres ejecuciones por repetición;
se informan mínimo, mediana y máximo por ejecución. Ambas versiones usan las mismas
filas, se calientan antes de medir y alternan el orden. Se incluyen 250, 2.080 y
las 27.633 filas reales de la base elegible. Los subconjuntos son prefijos de una
permutación con semilla 2026, no muestras comerciales representativas.

La medición sobre toda la base es **computacional**: no compara tarifas de estancias
distintas. Las comparaciones analíticas siguen limitadas al grupo principal.
La memoria se mide por separado con `tracemalloc`, tomando la mediana de tres picos.
Es memoria rastreada por Python, no RAM total ni toda la memoria nativa de NumPy/pandas
(Python Software Foundation, s. f.-b, s. f.-c).

In [7]:
mediciones, repeticiones = comparar_implementaciones(base)
display(mediciones)
assert mediciones['resumenes_equivalentes'].all()

,filas,grupos,version,tiempo_min_ms,tiempo_mediana_ms,tiempo_max_ms,pico_python_mediana_MiB,resumenes_equivalentes
0,250,30,bucle,31.491833,32.025467,32.565133,0.033397,True
1,250,30,vectorizada,1.644000,1.689300,1.769767,0.030116,True
2,2080,42,bucle,45.830333,46.041433,50.153867,0.113418,True
3,2080,42,vectorizada,1.924967,2.445733,2.594767,0.133609,True
4,27633,47,bucle,92.055133,92.487800,92.726700,1.721984,True
5,27633,47,vectorizada,6.105200,6.134300,7.367967,1.937831,True


## 8. Decisión de eficiencia, complejidad y límites

Con `n` filas y `k` grupos, filtrar toda la tabla por grupo añade trabajo del orden
de `n × k`; agrupar una vez evita esos filtros repetidos. Las medianas y los cuartiles
tienen además su propio costo: no se afirma que el cálculo completo sea O(n).
La implementación y el tamaño de los grupos afectan tiempo y memoria
(McKinney, 2022). Cambiar el tamaño también cambia la composición de las submuestras;
las mediciones no demuestran por sí solas una clase de complejidad asintótica.

Se adopta la versión con menor tiempo mediano sobre la base completa para los
resúmenes de F3. Se informa el compromiso con la memoria, sin prometer la misma razón
de rendimiento en otro equipo. Si los tiempos son casi iguales, la diferencia no
constituye evidencia robusta de superioridad.

Se utiliza **división funcional**, admitida por la guía: este flujo es tabular y su
secuencia de etapas es conocida. Introducir recursión agregaría llamadas y manejo de
profundidad sin resolver un problema anidado del proyecto.

In [8]:
reales = mediciones.loc[mediciones['filas'].eq(len(base))].set_index('version')
elegida = reales['tiempo_mediana_ms'].idxmin()
razon_tiempos = float(reales.loc['bucle', 'tiempo_mediana_ms'] / reales.loc['vectorizada', 'tiempo_mediana_ms'])
razon_memoria = float(reales.loc['vectorizada', 'pico_python_mediana_MiB'] / reales.loc['bucle', 'pico_python_mediana_MiB'])
decision = {'criterio': 'Menor tiempo mediano en la base completa; memoria informada por separado.',
            'version_elegida': elegida, 'filas_medidas': len(base),
            'razon_tiempo_bucle_sobre_vectorizada': razon_tiempos,
            'razon_memoria_vectorizada_sobre_bucle': razon_memoria,
            'alcance': 'Medición local; no establece superioridad universal.'}
calcular_resumen = {'bucle': resumen_por_bucle, 'vectorizada': resumen_vectorizado}[elegida]
resumenes = {v: calcular_resumen(principal, v) for v in ['barrio', 'tipo_alojamiento', 'capacidad']}
display(pd.DataFrame([decision]))
print(f'En este entorno, tiempo mediano bucle/vectorizada: {razon_tiempos:.2f}.')
print(f'Pico rastreado vectorizada/bucle: {razon_memoria:.2f}.')

,criterio,version_elegida,filas_medidas,razon_tiempo_bucle_sobre_vectorizada,razon_memoria_vectorizada_sobre_bucle,alcance
0,Menor tiempo mediano en la base completa; memo...,vectorizada,27633,15.077156,1.125348,Medición local; no establece superioridad univ...


En este entorno, tiempo mediano bucle/vectorizada: 15.08.
Pico rastreado vectorizada/bucle: 1.13.


## 9. Resultados locales para F4 y controles finales

F4 recibe la base elegible y los dos grupos ya procesados. No debe volver a limpiar,
imputar ni fijar otra moneda o ventana sin justificar y verificar ese cambio.
Los CSV por anuncio se escriben en `data/processed/`, fuera de Git; las tablas
publicadas contienen únicamente agregados. Se comprueba también que las referencias
de F2 continúan intactas después de la ejecución.

In [9]:
local = RAIZ / 'data/processed'
salidas = {'base_cotizaciones_f3.csv': base, 'grupo_principal_f3.csv': principal,
           'grupo_sensibilidad_f3.csv': sensibilidad}
for nombre, tabla in salidas.items():
    control = f2.exportar_y_verificar(tabla, local / nombre)
    assert control['cumple'].all()
f2.codificar_tipos(base).to_csv(local / 'indicadores_tipo_f3.csv', index=False, encoding='utf-8')
indicadores = pd.read_csv(local / 'indicadores_tipo_f3.csv', dtype={'id': 'string'})
pd.testing.assert_frame_equal(f2.codificar_tipos(base), indicadores, check_dtype=False)
_, huellas_despues = cargar_referencias_f2(RAIZ, RUTA_EVIDENCIA_F2)
pd.testing.assert_frame_equal(huellas_f2, huellas_despues, check_exact=True)
assert calcular_sha256(RUTA_ORIGINAL) == sha_original
assert huella_tabla(datos) == huella_original
validacion = pd.concat([validacion, pd.DataFrame([
    {'control': 'CSV_F3_exportados_y_releidos', 'cumple': True},
    {'control': 'indicadores_tipo_F3_releidos', 'cumple': True},
    {'control': 'referencias_F2_inmutables', 'cumple': True},
    {'control': 'resumenes_F2_F3_equivalentes', 'cumple': True},
])], ignore_index=True)
assert validacion['cumple'].all()
display(validacion)

,control,cumple
0,base_no_vacia,True
1,id_unico_y_presente,True
2,precio_finito_positivo,True
3,moneda_ARS,True
4,fechas_y_duracion_coherentes,True
5,capacidad_entera_positiva,True
6,grupos_principales_presentes,True
7,tipos_reconocidos,True
8,log_precio_reversible,True
9,razon_por_plaza_coherente,True


## 10. Pruebas, evidencias y cierre de la ejecución

La suite combina las 18 pruebas previas con las 11 de F3. Las pruebas adicionales
comprueban equivalencia, protección de parámetros, casos límite, detección de
alteraciones y funcionamiento de las mediciones. El registro conserva la versión
real del entorno y la referencia de Git consultada antes de generar resultados.
La ejecución puede dejar cambios sin confirmar: no se altera ese dato para ocultarlo.

In [10]:
pruebas = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'],
                         cwd=RAIZ, capture_output=True, text=True, encoding='utf-8', errors='replace')
salida_pruebas = pruebas.stdout + pruebas.stderr
(DESTINO / 'pruebas_f3.txt').write_text(salida_pruebas, encoding='utf-8')
print(salida_pruebas)
assert pruebas.returncode == 0, 'Revisar las pruebas antes de cerrar F3.'
conteo = re.search(r'Ran (\d+) tests?', salida_pruebas)
tablas = {'mediciones': mediciones, 'repeticiones': repeticiones, 'validacion': validacion,
          'casos_verificacion': casos, 'huellas_entradas_f2': huellas_f2,
          'faltantes': faltantes, 'transformaciones': transformaciones,
          'cambios_complementarios': cambios_complementarios,
          'etapas': pd.DataFrame(pipeline.registro_ajuste),
          **{'resumen_' + nombre: tabla for nombre, tabla in resumenes.items()}}
for nombre, tabla in tablas.items():
    tabla.to_csv(DESTINO / f'{nombre}.csv', index=False, encoding='utf-8')
evidencia = {
    'contexto': {'ambito': AMBITO_EJECUCION, 'python': platform.python_version(),
                 'pandas': pd.__version__, 'numpy': np.__version__, 'plataforma': platform.system(),
                 'git': git_inicio, 'sha256_original': sha_original},
    'resultados': {'base_elegible': len(base), 'grupo_principal': len(principal),
                   'grupo_adicional': len(sensibilidad)},
    'equivalencia': json.loads(equivalencia.to_json(orient='records')),
    'controles': json.loads(validacion.to_json(orient='records')),
    'parametros': pipeline.parametros, 'decision_eficiencia': decision,
    'pruebas': {'correctas': pruebas.returncode == 0,
                'cantidad': int(conteo.group(1)) if conteo else None},
    'sha256_salidas_locales': {nombre: calcular_sha256(local / nombre) for nombre in salidas},
}
(DESTINO / 'resultados_f3.json').write_text(
    json.dumps(evidencia, ensure_ascii=False, indent=2, allow_nan=False) + '\n', encoding='utf-8')
print(f'Comprobaciones satisfactorias: {int(validacion["cumple"].sum())}/{len(validacion)}.')
print('Evidencias generadas. Guardar el notebook con Ctrl+S antes de preparar el informe final.')

.............................
----------------------------------------------------------------------
Ran 29 tests in 1.282s

OK

Comprobaciones satisfactorias: 23/23.
Evidencias generadas. Guardar el notebook con Ctrl+S antes de preparar el informe final.


## 11. Referencias

Gamma, E., Helm, R., Johnson, R., & Vlissides, J. (1994). *Design patterns: Elements of
reusable object-oriented software*. Addison-Wesley.

Universidad Andrés Bello. (s. f.-a). *Guía de apoyo — Evaluación Sumativa 2 (Fase 3)* [Material docente].
https://github.com/magistercienciadatos/Material_Didactico_Semana2

Universidad Andrés Bello. (s. f.-b). *Apunte de la Fase 3: Núcleo algorítmico, eficiencia y programación orientada a objetos* [Material docente].
https://github.com/magistercienciadatos/Material_Didactico_Semana2

Python Software Foundation. (s. f.-a). *Classes*. Python documentation.
https://docs.python.org/3/tutorial/classes.html

Python Software Foundation. (s. f.-b). *timeit — Measure execution time of small code snippets*.
Python documentation. https://docs.python.org/3/library/timeit.html

Python Software Foundation. (s. f.-c). *tracemalloc — Trace memory allocations*.
Python documentation. https://docs.python.org/3/library/tracemalloc.html

The pandas development team. (s. f.). *pandas documentation*. https://pandas.pydata.org/docs/

McKinney, W. (2022). *Python for data analysis* (3rd ed.). O'Reilly Media.
https://wesmckinney.com/book/